**Setup of Patients List DB**

1- **Setup on BigQuery**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

# =========================================
# CONFIGURATION
# =========================================

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Patients_List"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# =========================================
# INITIALIZE BIGQUERY CLIENT
# =========================================

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# TABLE SCHEMA
# =========================================

schema = [

    # Primary Key
    bigquery.SchemaField(
        "Patient_U_ID",
        "STRING",
        mode="REQUIRED"
    ),

    # Basic Information
    bigquery.SchemaField("Patient_Name", "STRING"),
    bigquery.SchemaField("Phone_Number", "STRING"),
    bigquery.SchemaField("National_ID", "STRING"),
    bigquery.SchemaField("Address_Area", "STRING"),
    bigquery.SchemaField("Gender", "STRING"),

    # Measurements
    bigquery.SchemaField("Ht", "FLOAT"),
    bigquery.SchemaField("BMI", "FLOAT"),
    bigquery.SchemaField("BMI_Category", "STRING"),

    # Occupation
    bigquery.SchemaField("Occupation_Type", "STRING"),
    bigquery.SchemaField("Occupation_Details", "STRING"),

    # Lifestyle
    bigquery.SchemaField("Smoker", "BOOLEAN"),

    # Medical Information
    bigquery.SchemaField("Allergies", "STRING"),
    bigquery.SchemaField(
        "Chronic_Conditions",
        "STRING",
        mode="REPEATED"
    ),
    bigquery.SchemaField("Blood_Group", "STRING"),

    # Emergency Contact
    bigquery.SchemaField("Emergency_Contact", "STRING"),

    # Dates
    bigquery.SchemaField("Profile_Creation_Date", "DATE"),
    bigquery.SchemaField("First_Visit_Date", "DATE"),
    bigquery.SchemaField("Last_Visit_Date", "DATE")
]

# =========================================
# CREATE TABLE OBJECT
# =========================================

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

# =========================================
# CLUSTERING
# =========================================

table.clustering_fields = [
    "National_ID",
    "Patient_U_ID"
]

# =========================================
# CREATE TABLE
# =========================================

table = client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("Table created successfully")
print(TABLE_REF)
print("=================================")

Table created successfully
depihealthnux.Depihealthnux_Main.Patients_List


In [5]:
import sys
import psycopg2
sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT TO POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

cursor = conn.cursor()

# =========================================
# CREATE TABLE QUERY
# =========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS Patients_List (

    -- Primary Key
    Patient_U_ID TEXT PRIMARY KEY,

    -- Basic Information
    Patient_Name TEXT,
    Phone_Number TEXT,

    -- UNIQUE National ID
    National_ID TEXT UNIQUE,
    Address_Area TEXT,
    Gender TEXT,

    -- Measurements
    Ht FLOAT,
    BMI FLOAT,
    BMI_Category TEXT,

    -- Occupation
    Occupation_Type TEXT,
    Occupation_Details TEXT,

    -- Lifestyle
    Smoker BOOLEAN,

    -- Medical Information
    Allergies TEXT,
    Chronic_Conditions TEXT[],
    Blood_Group TEXT,

    -- Emergency Contact
    Emergency_Contact TEXT,

    -- Dates
    Profile_Creation_Date DATE,
    First_Visit_Date DATE,
    Last_Visit_Date DATE
);
"""

# =========================================
# EXECUTE QUERY
# =========================================

cursor.execute(create_table_query)

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Patients_List")
print("=================================")

# =========================================
# CLOSE CONNECTION
# =========================================

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Patients_List
